# Superstore Data Processing using PySpark

Objective:
To understand Spark DataFrames and perform data cleaning,
transformation and aggregation on Superstore dataset.

In [36]:
pip install pyspark

Note: you may need to restart the kernel to use updated packages.


In [119]:
import os

os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("CEI_Week5") \
    .getOrCreate()

print("Spark Ready")

Spark Ready


In [120]:
df = spark.read.csv(
    "Superstore_Modified.csv",
    header=True,
    inferSchema=True
)

In [121]:
#Load Dataset

df = spark.read.csv("Superstore_Modified.csv",header=True,inferSchema=True)
df.limit(6).toPandas()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,None,2,0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",None,3,0,219.582
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,None,2,0,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,None,5,0.45,-383.031
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,None,2,0.2,2.5164
5,6,CA-2014-115812,6/9/2014,6/14/2014,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,90032,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.86,7,0,14.1694


In [122]:
#Dataset Overview

print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 10004
Columns: 21


In [123]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

Check Schema

In [124]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)



Observation:

While inspecting the schema, Sales, Quantity and Discount
were inferred as string instead of numeric types.

These columns need type conversion before performing
aggregations and calculations.

In [132]:
#Schema Modification (Casting)

from pyspark.sql.types import DoubleType, IntegerType

df = df.withColumn(
    "Sales",
    col("Sales").cast(DoubleType())
)

df = df.withColumn(
    "Quantity",
    col("Quantity").cast(IntegerType())
)

df = df.withColumn(
    "Discount",
    col("Discount").cast(DoubleType())
)

In [133]:
#Verify Schema Again

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: string (nullable = true)



In [134]:
#Checking Null values

from pyspark.sql.functions import *

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,462,271,63,0


In [135]:
#Taking Average value

from pyspark.sql.functions import avg

df.select(avg("Sales")).show()

+------------------+
|        avg(Sales)|
+------------------+
|236.51481996182852|
+------------------+



In [136]:
#filling null values

mean_sales = df.select(avg("Sales")).collect()[0][0]

df = df.fillna({"Sales": mean_sales})

In [137]:
#checking again 

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,271,63,0


In [138]:
#Checking Duplicate Records

total_rows = df.count()

unique_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)

Total Rows: 9994
Unique Rows: 9994
Duplicate Rows: 0


In [139]:
#Removing Duplicate Rows

df = df.dropDuplicates()

In [140]:
print("Total Rows after removing duplicates:", df.count())

Total Rows after removing duplicates: 9994


Basic Statistics

In [142]:
#Category Wise Sales

from pyspark.sql.functions import sum

df.groupBy("Category") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|788107.9504958167|
|      Furniture|737828.9746430283|
|     Technology|837792.1855596944|
+---------------+-----------------+



In [144]:
#Aggregation

from pyspark.sql.functions import count, avg, min, max, round

df.agg(
    count("*").alias("Total_Rows"),
    round(avg("Sales"), 2).alias("Average_Sales"),
    round(min("Sales"), 2).alias("Minimum_Sales"),
    round(max("Sales"), 2).alias("Maximum_Sales")
).show()

+----------+-------------+-------------+-------------+
|Total_Rows|Average_Sales|Minimum_Sales|Maximum_Sales|
+----------+-------------+-------------+-------------+
|      9994|       236.51|         0.44|     22638.48|
+----------+-------------+-------------+-------------+



In [99]:
#Region Wise Sales

df.groupBy("Region") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .show()

+-------+------------------+
| Region|       Total_Sales|
+-------+------------------+
|  South|383004.82276093157|
|Central| 494500.9287999993|
|   East| 666950.5020000006|
|   West| 710003.0103152348|
+-------+------------------+



In [100]:
#Category Wise Profit

df.groupBy("Category") \
  .agg(sum("Profit").alias("Total_Profit")) \
  .show()

+---------------+------------------+
|       Category|      Total_Profit|
+---------------+------------------+
|Office Supplies|125760.18489999982|
|      Furniture|20065.238399999995|
|     Technology|       145422.0966|
+---------------+------------------+



In [101]:
#Top 10 Products by Sales

df.groupBy("Product Name") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .orderBy(col("Total_Sales").desc()) \
  .show(10, truncate=False)

+---------------------------------------------------------------------------+------------------+
|Product Name                                                               |Total_Sales       |
+---------------------------------------------------------------------------+------------------+
|Canon imageCLASS 2200 Advanced Copier                                      |61599.824         |
|Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind|27453.384000000002|
|Cisco TelePresence System EX90 Videoconferencing Unit                      |22638.48          |
|HON 5400 Series Task Chairs for Big and Tall                               |21870.576         |
|GBC DocuBind TL300 Electric Binding System                                 |19823.479         |
|GBC Ibimaster 500 Manual ProClick Binding System                           |19024.5           |
|Hewlett Packard LaserJet 3310 Copier                                       |18839.686         |
|"""HP Designjet T520 Inkjet L

In [102]:
#Top 10 Customers

df.groupBy("Customer Name") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .orderBy(col("Total_Sales").desc()) \
  .show(10, truncate=False)

+------------------+------------------+
|Customer Name     |Total_Sales       |
+------------------+------------------+
|Sean Miller       |25043.05          |
|Tamara Chand      |19017.848         |
|Raymond Buch      |15117.338999999998|
|Tom Ashbrook      |14595.62          |
|Adrian Barton     |14355.610999999999|
|Sanjit Chand      |14129.634000000002|
|Ken Lonsdale      |14071.917         |
|Hunter Lopez      |12873.297999999997|
|Sanjit Engle      |12209.438000000002|
|Christopher Conant|12129.072         |
+------------------+------------------+
only showing top 10 rows



In [148]:
from pyspark.sql.functions import count, sum, avg

df.groupBy("Category").agg(
    count("*").alias("Total_Orders"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales")
).show()

+---------------+------------+-----------------+------------------+
|       Category|Total_Orders|      Total_Sales|     Average_Sales|
+---------------+------------+-----------------+------------------+
|Office Supplies|        6026|788107.9504958167|130.78459185128057|
|      Furniture|        2121|737828.9746430283| 347.8684463192024|
|     Technology|        1847|837792.1855596944| 453.5962022521356|
+---------------+------------+-----------------+------------------+



In [103]:
#Filtering

from pyspark.sql.functions import col

df.filter(col("Sales") > 1000) \
  .select("Customer Name","Category","Sales","Profit") \
  .show(10, truncate=False)

+--------------------+---------------+--------+---------+
|Customer Name       |Category       |Sales   |Profit   |
+--------------------+---------------+--------+---------+
|Christopher Martinez|Office Supplies|6354.95 |3177.475 |
|Ken Lonsdale        |Technology     |1979.928|148.4946 |
|Kristen Hastings    |Technology     |3499.93 |909.9818 |
|Philip Fox          |Furniture      |1266.86 |291.3778 |
|Mitch Willingham    |Technology     |1919.976|215.9973 |
|Ted Trevino         |Furniture      |2404.704|150.294  |
|Ralph Arnett        |Technology     |1199.96 |224.9925 |
|Tonja Turnell       |Technology     |1718.4  |150.36   |
|Robert Waldorf      |Office Supplies|1924.16 |312.676  |
|Cindy Stewart       |Technology     |4499.985|-6599.978|
+--------------------+---------------+--------+---------+
only showing top 10 rows



In [104]:
df.filter(col("Category") == "Technology") \
  .select("Product Name","Category","Sales") \
  .show(10, truncate=False)

+---------------------------------------------------------------+----------+--------+
|Product Name                                                   |Category  |Sales   |
+---------------------------------------------------------------+----------+--------+
|AT&T 1070 Corded Phone                                         |Technology|445.96  |
|AT&T 841000 Phone                                              |Technology|207.0   |
|Microsoft Natural Ergonomic Keyboard 4000                      |Technology|47.984  |
|Plantronics Audio 995 Wireless Stereo Headset                  |Technology|263.88  |
|Mitel MiVoice 5330e IP Phone                                   |Technology|1979.928|
|Memorex Mini Travel Drive 16 GB USB 2.0 Flash Drive            |Technology|111.79  |
|Anker 24W Portable Micro USB Car Charger                       |Technology|35.168  |
|Logitech G500s Laser Gaming Mouse with Adjustable Weight Tuning|Technology|349.95  |
|SanDisk Ultra 32 GB MicroSDHC Class 10 Memory Card   

In [113]:
df.filter(col("Profit") < 0) \
  .select("Product Name","Sales","Profit") \
  .show(10, truncate=False)

+----------------------------------------------------------------------------+--------+----------+
|Product Name                                                                |Sales   |Profit    |
+----------------------------------------------------------------------------+--------+----------+
|Bretford CR4500 Series Slim Rectangular Table                               |NULL    |-383.031  |
|Holmes Replacement Filter for HEPA Air Cleaner, Very Large Room, HEPA Filter|68.81   |-123.858  |
|Storex DuraTech Recycled Plastic Frosted Binders                            |2.544   |-3.816    |
|Global Deluxe Stacking Chair, Gray                                          |71.372  |-1.0196   |
|Riverside Palais Royal Lawyers Bookcase, Royale Cherry Finish               |3083.43 |-1665.0522|
|Avery Recycled Flexi-View Covers for Binding Systems                        |9.618   |-7.0532   |
|Electrix Architect's Clamp-On Swing Arm Lamp, Black                         |190.92  |-147.963  |
|Atlantic 

In [146]:
#shuffle
from pyspark.sql.functions import sum

df.groupBy("Category") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|788107.9504958167|
|      Furniture|737828.9746430283|
|     Technology|837792.1855596944|
+---------------+-----------------+



The groupBy() operation is considered a wide transformation because Spark needs to move data across partitions before calculating the aggregation. This data movement process is called Shuffle.

In [106]:
#creating a new column

from pyspark.sql.functions import when

df = df.withColumn(
    "Profit_Status",
    when(col("Profit") > 0, "Profit")
    .otherwise("Loss")
)

df.select("Profit","Profit_Status").show(10)

+--------+-------------+
|  Profit|Profit_Status|
+--------+-------------+
| 12.0276|       Profit|
| 23.3928|       Profit|
|3177.475|       Profit|
|  55.745|       Profit|
| 10.3152|       Profit|
|   51.75|       Profit|
|  0.5998|         Loss|
|  1.2792|       Profit|
|  8.0997|       Profit|
|  4.4892|       Profit|
+--------+-------------+
only showing top 10 rows



In [107]:
#Group By + Revenue aggregation

from pyspark.sql.functions import sum, avg, max, min

df.groupBy("Category").agg(
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    max("Sales").alias("Max_Sales"),
    min("Sales").alias("Min_Sales")
).show()

+---------------+-----------------+-----------------+---------+---------+
|       Category|      Total_Sales|    Average_Sales|Max_Sales|Min_Sales|
+---------------+-----------------+-----------------+---------+---------+
|Office Supplies|697286.2596304683| 123.588489831703|  9892.74|    0.444|
|      Furniture|721272.9372456992|351.6689113825935| 4416.174|    1.892|
|     Technology|835900.0669999998|454.5405475802065| 22638.48|     0.99|
+---------------+-----------------+-----------------+---------+---------+



In [116]:
#Save Cleaned Data

pdf = df.toPandas()

pdf.to_csv(
    "Cleaned_Superstore.csv",
    index=False
)

print("File Saved Successfully")

File Saved Successfully


# Data Processing Pipeline

The complete data processing workflow followed in this assignment is shown below:

1. Loaded the Modified Superstore dataset into a PySpark DataFrame.
2. Examined the dataset structure using schema validation and record count.
3. Checked the dataset for missing (null) values.
4. Checked the dataset for duplicate records.
5. Handled missing values by replacing null entries with appropriate values.
6. Removed duplicate records to improve data quality.
7. Converted columns such as Sales, Quantity, and Discount to suitable numeric data types.
8. Performed aggregation operations using groupBy() and aggregate functions to generate business insights.
9. Applied filtering operations to analyze specific subsets of data, such as high-sales transactions and loss-making products.
10. Created a new derived column (Profit_Status) to classify records as Profit or Loss.
11. Generated summary statistics and business insights from the cleaned dataset.

Pipeline Flow:

Load Data -> Schema Validation -> Null Value Analysis -> Duplicate Record Analysis -> Missing Value Handling -> Duplicate Removal -> Data Type Conversion -> Aggregation & Analysis -> Filtering Operations -> Feature Creation (Profit_Status) -> Business Insights

In [149]:
pipeline_df = (
    df
    .dropDuplicates()
    .na.fill({"Sales": 0})
)

pipeline_df.groupBy("Category") \
    .agg(sum("Sales").alias("Total_Sales")) \
    .show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|788107.9504958167|
|      Furniture|737828.9746430283|
|     Technology|837792.1855596944|
+---------------+-----------------+

